# MCORE-1 Interactive Demo

**Metrical Core Representation v0.1** — ternary weight algebra, metrical tree validation, pattern generation, and encoding.

This notebook walks through each layer of the MCORE-1 spec with working code.

In [ ]:
from mcore_py import (
    Trit, Tension, Level, ProsodicUnit, Constituent, Budget,
    trit_add, trit_add_seq, tension_pair, complete, OVERFLOW,
    check_tree,
    encode_tme6, decode_tme6, Opcode,
    to_base64tme, from_base64tme,
    parse_mss, emit_mss,
)
from mcore_py.algebra import enumerate_patterns
from mcore_py.overlays import QuantitativeMetrics
from mcore_py.overlays.quantitative_metrics import (
    SyncopationType, resolve_heavy, classify_syllable
)
from mcore_py.renderers.terminal import render_scansion, render_terminal
from mcore_py.renderers.token_stream import render_token_stream, StreamValidator
from mcore_py.base64tme import annotate_stream, is_metrical_content

## 1. The Trit Algebra

Three weight states: **S1** (light), **S2** (heavy), **S3** (superheavy).  
Addition pools weight; overflow is an error, not a wrap-around.

In [ ]:
# Addition table
print("MCORE-1 Trit Addition Table")
print("="*35)
print(f"{'':>5} | {'S1':>5} {'S2':>5} {'S3':>5}")
print("-"*35)
for a in Trit:
    row = []
    for b in Trit:
        result = trit_add(a, b)
        row.append(result.name if result else '  OVF')
    print(f"{a.name:>5} | {row[0]:>5} {row[1]:>5} {row[2]:>5}")

In [ ]:
# Tension pairing: debt + surplus = resolved
print("Tension pairing:")
print(f"  SURPLUS + DEBT = {tension_pair(Trit.S1, Tension.SURPLUS, Trit.S1, Tension.DEBT)}")
print(f"  SURPLUS + SURPLUS = {tension_pair(Trit.S1, Tension.SURPLUS, Trit.S1, Tension.SURPLUS)}")

## 2. Building and Validating Metrical Trees

The fundamental invariant: **mora conservation**.  
Weight is a monoid homomorphism from tree structure to the trit algebra.

In [ ]:
# A valid trochaic foot: heavy + light = S3
trochee = Constituent(
    parent=ProsodicUnit(weight=Trit.S3, level=Level.L2_GANA, label="trochee"),
    children=[
        ProsodicUnit(weight=Trit.S2, level=Level.L1_AKSARA, label="-"),
        ProsodicUnit(weight=Trit.S1, level=Level.L1_AKSARA, label="u"),
    ],
)

result = check_tree(trochee)
print(f"Trochee (- u): {result}")
print(f"  Scansion: {render_scansion([c for c in trochee.children])}")

In [ ]:
# Intentional conservation violation: parent says S3, children sum to S2
bad_foot = Constituent(
    parent=ProsodicUnit(weight=Trit.S3, level=Level.L2_GANA),
    children=[
        ProsodicUnit(weight=Trit.S1, level=Level.L1_AKSARA),
        ProsodicUnit(weight=Trit.S1, level=Level.L1_AKSARA),
    ],
)

result = check_tree(bad_foot)
print(f"Bad foot: {result}")
for err in result.errors:
    print(f"  {err}")

## 3. Generalized Prastara (Pattern Completion)

Pingala's enumeration algorithm, extended from binary to ternary.  
Given *n* positions and a target budget, enumerate **all** valid patterns.

In [ ]:
# All 3-position patterns with total weight = S3 (value 2)
budget = Budget(min_weight=Trit.S3, max_weight=Trit.S3, exact=True)
patterns = enumerate_patterns(3, budget)

print(f"3 positions, budget S3 ({len(patterns)} patterns):")
print()

sym = {Trit.S1: 'u', Trit.S2: '–', Trit.S3: '≡'}
for p in patterns:
    print(f"  {' '.join(sym[t] for t in p)}  ({'+'.join(str(t.value) for t in p)}={sum(t.value for t in p)})")

In [ ]:
# Partial completion: first position fixed as S1, find all valid endings
partial = [Trit.S1, None, None]
results = complete(partial, budget)

print(f"Partial [S1, ?, ?] with budget S3 ({len(results)} completions):")
for r in results:
    print(f"  {' '.join(sym[t] for t in r)}")

## 4. TME-6 and Base64-TME Encoding

In [ ]:
# Encode a pattern
units = [
    ProsodicUnit(weight=Trit.S2, tension=Tension.NEUTRAL),
    ProsodicUnit(weight=Trit.S1, tension=Tension.NEUTRAL),
    ProsodicUnit(weight=Trit.S2, tension=Tension.DEBT),
    ProsodicUnit(weight=Trit.S1, tension=Tension.SURPLUS),
]

opcodes = encode_tme6(units)
ints = [op.value for op in opcodes]
b64 = to_base64tme(ints)

print(f"Pattern: {render_scansion(units)}")
print(f"TME-6:   {ints}")
print(f"Base64:  {b64}")
print(f"")
print("Annotated stream:")
for ch, val, name in annotate_stream(b64):
    print(f"  {ch}  ({val:2d})  {name}")
print(f"\nContains metrical content: {is_metrical_content(b64)}")

## 5. MSS Surface Syntax

In [ ]:
# Emit MSS for a unit
u = ProsodicUnit(weight=Trit.S2, tension=Tension.DEBT, level=Level.L2_GANA)
mss = emit_mss(u)
print(f"MSS: {mss}")

# Parse it back
from mcore_py.mss import parse_mss_to_units
parsed = parse_mss_to_units(mss)
print(f"Parsed: {parsed[0]}")

## 6. QuantitativeMetrics Overlay (Indo-European)

Syllable classification, iambic metron generation, resolution, syncopation.

In [ ]:
# Classify syllables
examples = [
    ("ta",  False, False, False),
    ("tat", False, True,  False),
    ("tā",  True,  False, False),
    ("tāt", True,  True,  False),
]

print(f"{'Syllable':>10} {'Long V':>7} {'Coda':>5} {'Type':>15} {'Weight':>7} {'Morae':>6}")
print("-"*60)
for name, vlong, coda, cluster in examples:
    st = classify_syllable(vlong, coda, cluster)
    from mcore_py.overlays.quantitative_metrics import syllable_weight, syllable_morae
    print(f"{name:>10} {str(vlong):>7} {str(coda):>5} {st.name:>15} {syllable_weight(st).name:>7} {syllable_morae(st):>6}")

In [ ]:
# Resolution: S2 -> S1 + S1
heavy = ProsodicUnit(weight=Trit.S2, tension=Tension.SURPLUS, label="resolved")
r1, r2 = resolve_heavy(heavy)
print(f"Resolution: {heavy.weight.name} -> {r1.weight.name} + {r2.weight.name}")
print(f"  Tension preserved: {r1.tension.name}, {r2.tension.name}")

In [ ]:
# Iambic metron with syncopation
for st in SyncopationType:
    metron = QuantitativeMetrics.iambic_metron(st)
    scansion = render_scansion(metron)
    tensions = [u.tension.name[0] for u in metron]
    print(f"  {st.name:>12}: {scansion}  tensions: {tensions}")

## 7. TokenStream Renderer (LLM Constrained Decoding)

Generate metrical tokens for interleaving with LLM text generation.

In [ ]:
# Generate a token stream for an iambic pattern
pattern = QuantitativeMetrics.iambic_metron(SyncopationType.NULL)
tokens = render_token_stream(pattern, foot_size=2)

print("Token stream for iambic metron (u – u –):")
for tok in tokens:
    if tok.is_boundary:
        print(f"  | BOUNDARY")
    else:
        print(f"  pos {tok.position}: {tok.weight.name}")

# Validate a sequence against the pattern
validator = StreamValidator(target=[t for t in tokens if not t.is_boundary])
sequence = [Trit.S1, Trit.S2, Trit.S1, Trit.S2]  # correct

print(f"\nValidating u – u –:")
for s in sequence:
    ok = validator.accept(s)
    print(f"  {s.name}: {'✓' if ok else '✗'}")
print(f"Complete: {validator.complete}")

---

**MCORE-1 v0.1** · Jacob Walker · [github.com/vortexpixelz/mcore1](https://github.com/vortexpixelz/mcore1)